# Tekne Dedektörü v5 — Google Colab GPU Eğitimi

Bu notebook, Mac'te (M4, MPS) çok yavaş kalan `train_v5.py` eğitimini (YOLOv8s, imgsz=960) Colab'ın ücretsiz GPU'sunda (genelde T4, 16GB VRAM) çalıştırır.

**Önce yapman gerekenler:**
1. Üstteki menüden **Çalışma zamanı (Runtime) > Çalışma zamanı türünü değiştir > T4 GPU** seç.
2. `boat_v5_colab_bundle.zip` dosyasını (Mac'indeki `depth-anything` klasöründe, 313MB) Google Drive'ına yükle — en kolayı `My Drive` kökü. Drive web arayüzünden sürükle-bırak yeterli, internet hızına göre birkaç dakika sürer.
3. Aşağıdaki hücreleri sırayla çalıştır.

In [ ]:
# 1) GPU kontrolü — çıktıda bir Tesla T4 (veya benzeri) görmelisin
!nvidia-smi

In [ ]:
# 2) Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Zip dosyasını Drive'dan al ve aç
# Zip'i Drive'da farklı bir yere yüklediysen ZIP_PATH'i güncelle.
ZIP_PATH = '/content/drive/MyDrive/boat_v5_colab_bundle.zip'

!mkdir -p /content/work
!unzip -q "$ZIP_PATH" -d /content/work
!ls /content/work

In [ ]:
# 4) ultralytics kur
!pip install -q ultralytics

In [ ]:
# 5) dataset.yaml içindeki path'i Colab'daki yeni konuma göre düzelt
# (Mac'te: /Users/armin/Desktop/depth-anything/yolo_dataset_v4 idi)
import pathlib

yaml_path = pathlib.Path('/content/work/yolo_dataset_v4/dataset.yaml')
content = yaml_path.read_text()
print('--- eski ---')
print(content)

new_content = content.replace(
    '/Users/armin/Desktop/depth-anything/yolo_dataset_v4',
    '/content/work/yolo_dataset_v4'
)
yaml_path.write_text(new_content)
print('--- yeni ---')
print(yaml_path.read_text())

In [ ]:
# 6) Eğitim — train_v5.py ile aynı hiperparametreler, device='mps' yerine device=0 (GPU)
# T4'te imgsz=960 + batch=16, Mac'teki gibi disk takasına düşmeden rahat çalışır.
# Eğitim koparsa (Colab oturumu zaman aşımına uğrarsa), aynı hücreyi resume=True ile
# tekrar çalıştırarak en son checkpoint'ten devam edebilirsin (7. hücreye bak).
from ultralytics import YOLO

model = YOLO('/content/work/yolov8s.pt')
results = model.train(
    data='/content/work/yolo_dataset_v4/dataset.yaml',
    epochs=80,
    imgsz=960,
    device=0,
    batch=16,
    patience=40,
    project='/content/work/runs_boat_yolo',
    name='boat_v5_s',
    verbose=True,
)

In [ ]:
# 7) Eğitim koptuysa devam ettirmek için (6. hücre yerine bunu çalıştır):
# from ultralytics import YOLO
# model = YOLO('/content/work/runs_boat_yolo/boat_v5_s/weights/last.pt')
# results = model.train(resume=True)

In [ ]:
# 8) Bitince: sonuçları (weights + results.csv + grafikler) Drive'a kopyala ki
# Colab oturumu kapansa bile kaybolmasın.
!mkdir -p /content/drive/MyDrive/boat_v5_results
!cp -r /content/work/runs_boat_yolo/boat_v5_s /content/drive/MyDrive/boat_v5_results/
print('Kopyalandı: Google Drive > boat_v5_results > boat_v5_s')

## Eğitim bitince Mac'ine geri alma

1. Drive'daki `boat_v5_results/boat_v5_s` klasörünü indir (özellikle `weights/best.pt` ve `weights/last.pt` lazım — geri kalanı, grafikler/results.csv, isteğe bağlı).
2. Mac'inde `depth-anything/runs_boat_yolo/boat_v5_s/` altına yerleştir (klasör yoksa oluştur).
3. `run.py` veya test scriptlerinde bu yeni `best.pt`'yi kullan.

**Not — Colab'ın ücretsiz kotası:** Oturumlar genelde ~12 saatte bir kesilir ve uzun süre etkileşimsiz kalırsan (sekmeyi kapatmak, ~30-90 dk boşta kalmak) daha erken de kopabilir. 80 epoch imgsz=960'ta GPU'da muhtemelen 2-4 saat sürer (T4'e göre değişir) — tek oturumda bitmesi büyük olasılıkla mümkün, ama koparsa 7. hücredeki `resume=True` ile devam edebilirsin. Sekmeyi açık tutmak ve arada bir tıklamak kopmaları azaltır.